# TRELLIS.2 on A100 — v5 (Working)

## Why so many patches?
TRELLIS.2 is a research model released by Microsoft in early 2025. It was designed to run in a specific environment that doesn't match Colab's setup out of the box. Specifically:

1. **Missing `sparse_structure_decoder`** — TRELLIS.2 references a checkpoint from the original TRELLIS repo, not included in the TRELLIS.2 download
2. **Gated models** — DINOv3 and RMBG-2.0 require HuggingFace access approval
3. **dtype mismatches** — The model internally uses bfloat16 but Colab's PyTorch operations default to float32/float16, causing crashes throughout
4. **Custom CUDA extensions** — FlexGEMM, CuMesh, o-voxel, nvdiffrast must be compiled from source (20+ mins first time)
5. **`manual_cast` bug** — A utility function silently skips dtype casting when PyTorch autocast is active, causing NaN values
6. **Attention backend** — Sparse attention requires xformers; dense attention works with sdpa
7. **`sparse_structure_decoder` incompatibility** — The decoder from original TRELLIS produces all-negative values with TRELLIS.2's flow model, requiring a percentile threshold instead of > 0

None of these are bugs you introduced — they are gaps between the research release and a production-ready setup.

## Session flow

### First time (~35 mins)
Cell 1 → 2 → 3 → 4 → **Restart session** → 2 → 3 → 5 → 6 → 7 → 8 → 9

### On reconnect (~8 mins)
Cell 1 → 2 → 3 → 4 → **Restart session** → 2 → 3 → 6 → 7 → 8 → 9

### Why restart after Cell 4?
Cell 4 patches source files on disk. Python caches already-imported modules so patches only take effect after a fresh import via restart.

## Prerequisites
- Runtime → Change runtime type → **A100 GPU**
- HuggingFace token with 'Read access to public gated repos'
- Access approved for `facebook/dinov3-vitl16-pretrain-lvd1689m` (request at huggingface.co)

## Cell 1 — Verify GPU

In [ ]:
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 2 — HuggingFace Login

In [ ]:
from huggingface_hub import login
login()

## Cell 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/drive')
print('✅ Drive mounted')

## Cell 4 — Install Dependencies & Apply Source Patches
**Run once per session. Restart runtime when complete.**
Restores from Drive if available (~5 mins), otherwise builds from scratch (~35 mins).

In [ ]:
import os, glob

DRIVE_EXT = '/drive/MyDrive/TRELLIS2_extensions'
os.environ['TORCH_CUDA_ARCH_LIST'] = '8.0'
os.environ['FORCE_CUDA'] = '1'

# ── Clone TRELLIS.2 ───────────────────────────────────────────────────────────
if not os.path.exists('/content/TRELLIS.2') or not os.path.exists('/content/TRELLIS.2/trellis2'):
    print('📦 Cloning TRELLIS.2...')
    os.chdir('/content')
    os.system('rm -rf TRELLIS.2')
    os.system('git clone --recurse-submodules https://github.com/microsoft/TRELLIS.2.git TRELLIS.2')
    print('✅ Cloned')
else:
    print('✅ TRELLIS.2 already exists')

# ── Install extensions ────────────────────────────────────────────────────────
os.makedirs('/tmp/extensions', exist_ok=True)

# Always install these from pip/git (fast, no Drive needed)
print('📦 Installing base packages...')
os.system('pip install -q xformers kornia timm imageio imageio-ffmpeg tqdm easydict opencv-python-headless ninja trimesh transformers gradio==6.0.1 tensorboard pandas lpips zstandard')
os.system('pip install -q git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8')

# nvdiffrast — install from git (fast)
print('📦 Installing nvdiffrast...')
os.system('pip install -q git+https://github.com/NVlabs/nvdiffrast.git --no-build-isolation')

# FlexGEMM, CuMesh, o-voxel — restore from Drive or build
for ext, git_url, extra in [
    ('FlexGEMM', 'https://github.com/JeffreyXiang/FlexGEMM.git', '--recursive'),
    ('CuMesh',   'https://github.com/JeffreyXiang/CuMesh.git',   '--recursive'),
]:
    local = f'/tmp/extensions/{ext}'
    drive_path = f'{DRIVE_EXT}/{ext}'
    if not os.path.exists(local):
        if os.path.exists(drive_path):
            print(f'💾 Restoring {ext} from Drive...')
            os.system(f'cp -r {drive_path} {local}')
        else:
            print(f'🛠️ Building {ext}...')
            os.system(f'git clone {git_url} {local} {extra} -q')
    os.system(f'pip install -q {local} --no-build-isolation')
    print(f'  ✅ {ext}')

# o-voxel special case (in repo)
ovoxel_local = '/tmp/o-voxel'
if not os.path.exists(ovoxel_local):
    os.system('cp -r /content/TRELLIS.2/o-voxel /tmp/o-voxel')
os.system('pip install /tmp/o-voxel --no-build-isolation -q')
print('  ✅ o-voxel')

# nvdiffrec
nvdiffrec = '/tmp/extensions/nvdiffrec'
if not os.path.exists(nvdiffrec):
    os.system(f'git clone -b renderutils https://github.com/JeffreyXiang/nvdiffrec.git {nvdiffrec} -q')
os.system(f'pip install -q {nvdiffrec} --no-build-isolation')
print('  ✅ nvdiffrec')

# ── SOURCE FILE PATCHES ───────────────────────────────────────────────────────
print('\n🔧 Applying source patches...')

# Patch 1: manual_cast — always cast regardless of autocast state
utils_file = '/content/TRELLIS.2/trellis2/modules/utils.py'
with open(utils_file) as f: src = f.read()
old1 = 'def manual_cast(tensor, dtype):\n    """\n    Cast if autocast is not enabled.\n    """\n    if not torch.is_autocast_enabled():\n        return tensor.type(dtype)\n    return tensor'
new1 = 'def manual_cast(tensor, dtype):\n    """\n    Always cast to dtype regardless of autocast state.\n    """\n    return tensor.to(dtype)'
if old1 in src:
    with open(utils_file, 'w') as f: f.write(src.replace(old1, new1))
    print('✅ Patch 1: manual_cast')
else:
    print('✅ Patch 1: manual_cast already applied')

# Patch 2: timestep_embedding — preserve input dtype
for fpath in glob.glob('/content/TRELLIS.2/trellis2/models/**/*.py', recursive=True):
    with open(fpath) as f: src = f.read()
    if 'args = t[:, None].float() * freqs[None]' in src:
        old2 = '        ).to(device=t.device)\n        args = t[:, None].float() * freqs[None]'
        new2 = '        ).to(device=t.device, dtype=t.dtype)\n        args = t[:, None] * freqs[None]'
        with open(fpath, 'w') as f: f.write(src.replace(old2, new2))
        print(f'✅ Patch 2: timestep_embedding in {os.path.basename(fpath)}')

# Patch 3: BiRefNet wrapper — cast input to model dtype
rembg_file = '/content/TRELLIS.2/trellis2/pipelines/rembg/BiRefNet.py'
with open(rembg_file) as f: src = f.read()
old3 = '        input_images = self.transform_image(image).unsqueeze(0).to("cuda")'
new3 = '        input_images = self.transform_image(image).unsqueeze(0).to("cuda")\n        input_images = input_images.to(next(self.model.parameters()).dtype)'
if old3 in src:
    with open(rembg_file, 'w') as f: f.write(src.replace(old3, new3))
    print('✅ Patch 3: BiRefNet wrapper')
else:
    print('✅ Patch 3: BiRefNet wrapper already applied')

print('\n✅ All done!')
print('⚠️  RESTART RUNTIME NOW: Runtime → Restart session')
print('   Then: Cell 2 → Cell 3 → Cell 5 → Cell 6 → Cell 7 → Cell 8 → Cell 9')

## Cell 5 — Download Model Weights (~12GB)
Skip if already on Drive.

In [ ]:
import os
from huggingface_hub import snapshot_download, hf_hub_download

DRIVE_MODEL = '/drive/MyDrive/TRELLIS.2-4B'
LOCAL_MODEL = '/content/models/TRELLIS.2-4B'

if os.path.exists(DRIVE_MODEL):
    print(f'✅ Model on Drive: {DRIVE_MODEL}')
else:
    print('Downloading TRELLIS.2-4B (~12GB)...')
    snapshot_download(repo_id='microsoft/TRELLIS.2-4B', local_dir=LOCAL_MODEL)
    print('✅ Downloaded')

# Download sparse_structure_decoder from original TRELLIS
SS_DEC_DIR = '/content/models/TRELLIS-extra/ckpts'
os.makedirs(SS_DEC_DIR, exist_ok=True)
if not os.path.exists(f'{SS_DEC_DIR}/ss_dec_conv3d_16l8_fp16.safetensors'):
    print('Downloading sparse_structure_decoder from original TRELLIS...')
    for fname in ['ckpts/ss_dec_conv3d_16l8_fp16.json', 'ckpts/ss_dec_conv3d_16l8_fp16.safetensors']:
        hf_hub_download(repo_id='microsoft/TRELLIS-image-large', filename=fname, local_dir='/content/models/TRELLIS-extra')
    print('✅ Downloaded sparse_structure_decoder')
else:
    print('✅ sparse_structure_decoder already present')

## Cell 6 — Patch BiRefNet HF Cache
Run every session.

In [ ]:
import glob

cache_files = glob.glob('/root/.cache/huggingface/modules/transformers_modules/ZhengPeng7/BiRefNet/*/birefnet.py')
if cache_files:
    with open(cache_files[0]) as f: src = f.read()
    replacements = [
        ('x, H, W = self.patch_embed1(x)', 'x = x.to(next(self.patch_embed1.parameters()).dtype)\n        x, H, W = self.patch_embed1(x)'),
        ('x, H, W = self.patch_embed2(x)', 'x = x.to(next(self.patch_embed2.parameters()).dtype)\n        x, H, W = self.patch_embed2(x)'),
        ('x, H, W = self.patch_embed3(x)', 'x = x.to(next(self.patch_embed3.parameters()).dtype)\n        x, H, W = self.patch_embed3(x)'),
        ('x, H, W = self.patch_embed4(x)', 'x = x.to(next(self.patch_embed4.parameters()).dtype)\n        x, H, W = self.patch_embed4(x)'),
        ('x = self.patch_embed(x)',         'x = x.to(next(self.patch_embed.parameters()).dtype)\n        x = self.patch_embed(x)'),
    ]
    patched = sum(1 for old, new in replacements if old in src and new not in src)
    if patched:
        for old, new in replacements:
            if old in src and new not in src: src = src.replace(old, new)
        with open(cache_files[0], 'w') as f: f.write(src)
        print(f'✅ Patched birefnet.py ({patched} locations)')
    else:
        print('✅ birefnet.py already patched')
else:
    print('ℹ️  BiRefNet cache not found yet — will patch after Cell 7 downloads it')

## Cell 7 — Load Pipeline
Run every session after restart.

In [ ]:
import os, sys, json, types, torch, importlib, glob

# ── Capture TRUE originals BEFORE any patching ────────────────────────────────
import torch.nn.modules.conv as _conv_mod
_TRUE_CONV3D    = _conv_mod.Conv3d._conv_forward
_TRUE_LAYER_NORM = torch.layer_norm
_TRUE_LINEAR    = torch.nn.Linear.forward
print('✅ True originals captured')

# ── Clear cached trellis modules ──────────────────────────────────────────────
to_remove = [k for k in sys.modules if 'trellis' in k.lower()]
for k in to_remove: del sys.modules[k]
print(f'✅ Cleared {len(to_remove)} cached trellis modules')

# ── Paths ─────────────────────────────────────────────────────────────────────
code_root  = '/content/TRELLIS.2'
model_path = '/drive/MyDrive/TRELLIS.2-4B' if os.path.exists('/drive/MyDrive/TRELLIS.2-4B') else '/content/models/TRELLIS.2-4B'
SS_DEC     = '/content/models/TRELLIS-extra/ckpts/ss_dec_conv3d_16l8_fp16'
print(f'Model: {model_path}')

# ── Stubs — only for o_voxel ──────────────────────────────────────────────────
def make_pkg(name):
    m = types.ModuleType(name); m.__path__ = []; m.__package__ = name
    sys.modules[name] = m; return m
def make_mod(full_name, parent=None):
    m = types.ModuleType(full_name); sys.modules[full_name] = m
    if parent: setattr(parent, full_name.split('.')[-1], m)
    return m

if 'o_voxel' not in sys.modules:
    o_voxel = make_pkg('o_voxel')
    ov_c = make_mod('o_voxel.convert', o_voxel)
    ov_c.flexible_dual_grid_to_mesh = lambda *a, **kw: None

# ── Verify real flex_gemm ─────────────────────────────────────────────────────
try:
    import flex_gemm
    assert hasattr(flex_gemm.ops.spconv, 'set_algorithm')
    print('✅ Real flex_gemm loaded')
except Exception as e:
    print(f'❌ flex_gemm error: {e} — installing...')
    os.system('pip install /tmp/extensions/FlexGEMM --no-build-isolation -q')
    import importlib; flex_gemm = importlib.import_module('flex_gemm')

# ── Paths & env ───────────────────────────────────────────────────────────────
for p in [code_root, f'{code_root}/trellis2']:
    if p not in sys.path: sys.path.insert(0, p)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ['ATTN_BACKEND'] = 'xformers'

# ── Runtime patches ───────────────────────────────────────────────────────────
def _safe_linear(self, x):
    if x.dtype != self.weight.dtype: x = x.to(self.weight.dtype)
    return _TRUE_LINEAR(self, x)
torch.nn.Linear.forward = _safe_linear

def _safe_ln(input, normalized_shape, weight=None, bias=None, eps=1e-5, cudnn_enable=True):
    dt = input.dtype
    return _TRUE_LAYER_NORM(input.float(), normalized_shape,
                            weight.float() if weight is not None else None,
                            bias.float() if bias is not None else None,
                            eps, cudnn_enable).to(dt)
torch.layer_norm = _safe_ln

def _safe_conv3d(self, input, weight, bias):
    if input.dtype != weight.dtype: input = input.to(weight.dtype)
    return _TRUE_CONV3D(self, input, weight, bias)
torch.nn.Conv3d._conv_forward = _safe_conv3d
print('✅ Linear, layer_norm, Conv3d patched')

# ── Patch transformers ────────────────────────────────────────────────────────
import transformers.utils.loading_report as _lr
import transformers.modeling_utils as _mu
importlib.reload(_lr); importlib.reload(_mu)
_real_log = _lr.log_state_dict_report
def _patched_log(*args, **kwargs):
    try: return _real_log(*args, **kwargs)
    except RuntimeError as e:
        if 'ignore_mismatched_sizes' in str(e): return
        raise
_lr.log_state_dict_report = _patched_log
_mu.log_state_dict_report = _patched_log
print('✅ Transformers patch applied')

# ── Patch pipeline.json ───────────────────────────────────────────────────────
with open(f'{model_path}/pipeline.json') as f: cfg = json.load(f)
cfg['args']['models']['sparse_structure_decoder'] = SS_DEC
cfg['args']['rembg_model']['args']['model_name'] = 'ZhengPeng7/BiRefNet'
with open(f'{model_path}/pipeline_patched.json', 'w') as f: json.dump(cfg, f, indent=2)
print('✅ Config patched')

# ── Load pipeline ─────────────────────────────────────────────────────────────
print('\n🚀 Loading pipeline (2-5 mins)...')
try:
    from trellis2.pipelines import Trellis2ImageTo3DPipeline
    pipeline = Trellis2ImageTo3DPipeline.from_pretrained(model_path, config_file='pipeline_patched.json')
    pipeline.to('cuda')
    pipeline.low_vram = False

    # Cast all to bfloat16
    for name, model in pipeline.models.items():
        pipeline.models[name] = model.to('cuda').to(torch.bfloat16)
        if hasattr(model, 'dtype'): model.dtype = torch.bfloat16
    pipeline.image_cond_model.model = pipeline.image_cond_model.model.to('cuda').to(torch.bfloat16)
    pipeline.rembg_model.model = pipeline.rembg_model.model.to('cuda').float()
    print('✅ Models on CUDA (bfloat16, rembg fp32)')

    # Fix flex_gemm reference
    import trellis2.modules.sparse.conv.conv_flex_gemm as conv_mod
    conv_mod.sparse_conv3d_forward.__globals__['flex_gemm'] = flex_gemm
    print('✅ flex_gemm reference fixed')

    # Fix o_voxel reference
    import o_voxel as _real_ovoxel
    from o_voxel.convert import flexible_dual_grid_to_mesh as _real_fdg
    import trellis2.models.sc_vaes.fdg_vae as fdg_mod
    fdg_mod.flexible_dual_grid_to_mesh = _real_fdg
    print('✅ o_voxel reference fixed')

    # Set attention backends
    import trellis2.modules.sparse.config as sparse_config
    sparse_config.set_attn_backend('xformers')
    import trellis2.modules.attention.config as attn_config
    attn_config.BACKEND = 'sdpa'
    print('✅ Attention: sparse=xformers, dense=sdpa')

    # Patch sample_sparse_structure with percentile threshold
    # Note: sparse_structure_decoder from original TRELLIS produces all-negative
    # values with TRELLIS.2 flow model output, so we use top 50% threshold
    def patched_sample_sparse_structure(self, cond, resolution, num_samples=1, sampler_params={}):
        flow_model = self.models['sparse_structure_flow_model']
        noise = torch.randn(num_samples, flow_model.in_channels,
                            *[flow_model.resolution]*3).to('cuda').to(torch.bfloat16)
        sampler_params = {**self.sparse_structure_sampler_params, **sampler_params}
        z_s = self.sparse_structure_sampler.sample(
            flow_model, noise, **cond, **sampler_params,
            verbose=True, tqdm_desc='Sampling sparse structure'
        ).samples
        decoder = self.models['sparse_structure_decoder']
        z_s = z_s.to(next(decoder.parameters()).dtype)
        decoded = decoder(z_s)
        threshold = torch.quantile(decoded.float(), 0.5)
        decoded = decoded > threshold
        if resolution != decoded.shape[2]:
            ratio = decoded.shape[2] // resolution
            decoded = torch.nn.functional.max_pool3d(decoded.float(), ratio, ratio, 0) > 0.5
        coords = torch.argwhere(decoded)[:, [0,2,3,4]].int()
        print(f'Voxels: {coords.shape[0]}')
        return coords
    pipeline.sample_sparse_structure = types.MethodType(patched_sample_sparse_structure, pipeline)
    print('✅ sample_sparse_structure patched')

    # Re-patch birefnet cache if available
    cache_files = glob.glob('/root/.cache/huggingface/modules/transformers_modules/ZhengPeng7/BiRefNet/*/birefnet.py')
    if cache_files:
        with open(cache_files[0]) as f: src = f.read()
        replacements = [
            ('x, H, W = self.patch_embed1(x)', 'x = x.to(next(self.patch_embed1.parameters()).dtype)\n        x, H, W = self.patch_embed1(x)'),
            ('x, H, W = self.patch_embed2(x)', 'x = x.to(next(self.patch_embed2.parameters()).dtype)\n        x, H, W = self.patch_embed2(x)'),
            ('x, H, W = self.patch_embed3(x)', 'x = x.to(next(self.patch_embed3.parameters()).dtype)\n        x, H, W = self.patch_embed3(x)'),
            ('x, H, W = self.patch_embed4(x)', 'x = x.to(next(self.patch_embed4.parameters()).dtype)\n        x, H, W = self.patch_embed4(x)'),
            ('x = self.patch_embed(x)',         'x = x.to(next(self.patch_embed.parameters()).dtype)\n        x = self.patch_embed(x)'),
        ]
        patched = sum(1 for old, new in replacements if old in src and new not in src)
        if patched:
            for old, new in replacements:
                if old in src and new not in src: src = src.replace(old, new)
            with open(cache_files[0], 'w') as f: f.write(src)
            for k in list(sys.modules.keys()):
                if 'birefnet' in k.lower(): del sys.modules[k]
            print(f'✅ birefnet.py cache patched')

    print('\n✅ Pipeline ready!')
except Exception as e:
    import traceback; traceback.print_exc()

## Cell 8 — Run Inference

In [ ]:
from PIL import Image
from google.colab import files
import torch, gc

uploaded = files.upload()
image = Image.open(list(uploaded.keys())[0]).convert('RGB')
print(f'Loaded: {image.size}')

# Clear CUDA cache before running
gc.collect()
torch.cuda.empty_cache()

print('\n🎨 Generating 3D model...')
with torch.no_grad():
    outputs = pipeline.run(
        image,
        pipeline_type='512',  # Options: '512', '1024', '1024_cascade', '1536_cascade'
    )

print('\n✅ Generation complete!')
print(f'Got {len(outputs)} output(s): {[type(o).__name__ for o in outputs]}')

## Cell 9 — Export & Download GLB

In [ ]:
import os, numpy as np, trimesh
from google.colab import files

os.makedirs('/content/outputs', exist_ok=True)
mesh = outputs[0]

print('Mesh info:')
print(f'  Vertices: {len(mesh.vertices)}')
print(f'  Faces: {len(mesh.faces)}')
print(f'  Attrs shape: {mesh.attrs.shape}')
print(f'  Layout: {mesh.layout}')

vertices = mesh.vertices.cpu().float().numpy()
faces = mesh.faces.cpu().numpy()
attrs = mesh.attrs.cpu().float().numpy()

# Export geometry-only GLB (always works)
tm = trimesh.Trimesh(vertices=vertices, faces=faces)
path_geo = '/content/outputs/output_geometry.glb'
tm.export(path_geo)
print(f'\n✅ Geometry exported: {os.path.getsize(path_geo)/1e6:.1f} MB')
files.download(path_geo)

# Try to export with colors
try:
    base_color = np.clip(attrs[:, mesh.layout['base_color']], 0, 1)
    roughness  = np.clip(attrs[:, mesh.layout['roughness']], 0, 1)
    metallic   = np.clip(attrs[:, mesh.layout['metallic']], 0, 1)

    # Per-vertex colors via face averaging
    n_verts = len(vertices)
    vert_colors = np.zeros((n_verts, 3))
    vert_counts = np.zeros(n_verts)
    for fi, face in enumerate(faces):
        color = base_color[fi] if fi < len(base_color) else [0.5,0.5,0.5]
        for vi in face:
            vert_colors[vi] += color
            vert_counts[vi] += 1
    vert_counts = np.maximum(vert_counts, 1)
    vert_colors = vert_colors / vert_counts[:, None]
    vert_colors_rgba = np.concatenate([vert_colors, np.ones((n_verts,1))], axis=1)

    tm_colored = trimesh.Trimesh(
        vertices=vertices, faces=faces,
        vertex_colors=(vert_colors_rgba * 255).astype(np.uint8)
    )
    path_colored = '/content/outputs/output_colored.glb'
    tm_colored.export(path_colored)
    print(f'✅ Colored exported: {os.path.getsize(path_colored)/1e6:.1f} MB')
    files.download(path_colored)
except Exception as e:
    print(f'⚠️  Color export failed: {e}')
    print('   Geometry-only GLB was already downloaded above.')